# 03 · Delegation, Hands-off, and A2A Runtime

Harness orchestrator가 직접 답을 만들지 않고 `researcher`에 조사 작업을 **delegation**한 뒤, evidence packet을 `analyst_writer`에게 **handoff**하는 실행 가능한 A2A 예제입니다. 외부 모델·MCP·업무 시스템을 호출하지 않는 in-memory transport이므로 메시지 경로와 권한 경계를 안전하게 확인할 수 있습니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124;">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️ UI 실행</div>
  <h2 style="margin: 8px 0; font-size: 1.05em; color: #202124;">UI: Harness delegation &amp; A2A trace</h2>
  <p style="margin: 7px 0; line-height: 1.6;">요청을 보내면 orchestrator → researcher → analyst_writer → user 경로를 실제로 dispatch합니다. 아래 trace에는 각 message의 sender, recipient, type, correlation ID가 표시됩니다.</p>
</div>

In [ ]:
from pathlib import Path
import sys
import gradio as gr

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '5_harness').is_dir())
sys.path.insert(0, str(project_root / '5_harness'))
from src.delegation_a2a import run_delegation_a2a

def run_a2a_ui(request: str):
    outcome = run_delegation_a2a(request)
    return outcome['report']['summary'], outcome['verification'], outcome['trace']

with gr.Blocks() as a2a_ui:
    gr.Markdown('## Harness Delegation and A2A Playground')
    gr.Markdown('이 실행은 fixture를 읽기만 하며 외부 시스템 쓰기 작업을 수행하지 않습니다.')
    request = gr.Textbox(label='Request', value='재고를 조사해 발주 검토 보고서를 작성해줘.')
    run = gr.Button('Delegate and run A2A', variant='primary')
    report = gr.Textbox(label='Final report')
    verification = gr.JSON(label='A2A verification')
    trace = gr.JSON(label='Delivered message trace')
    run.click(run_a2a_ui, inputs=request, outputs=[report, verification, trace])
a2a_ui.launch(inline=True, share=False, show_error=True, server_name='127.0.0.1', height=900)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124;">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅ 체크포인트</div>
  <ul style="margin: 8px 0 0; padding-left: 1.3em; line-height: 1.6;">
    <li>검증 결과가 <code>passed: true</code>이고 세 hop이 순서대로 보인다.</li>
    <li>모든 메시지가 같은 <code>correlation_id</code>를 가진다.</li>
    <li>researcher의 evidence packet만 analyst_writer가 최종 보고서로 변환한다.</li>
  </ul>
</div>

---

## 선택 심화: 코드로 contract와 실행 결과 확인

이 예제의 A2A는 네트워크 endpoint를 사용하는 특정 벤더 SDK가 아니라, agent 사이에서 실제로 전달·검증되는 메시지 envelope를 보여 줍니다. 원격 A2A transport로 교체할 때도 `message_id`, `correlation_id`, `sender`, `recipient`, `message_type`, `payload` 계약과 역할 경계를 유지해야 합니다.

### 1. Delegation과 handoff의 책임 분리

- Orchestrator: 작업을 researcher에 위임만 합니다.
- Researcher: 읽기 전용 근거를 `evidence_packet`으로 만듭니다. 사용자 답변을 직접 만들지 않습니다.
- Analyst Writer: evidence packet만 받아 최종 report를 작성합니다.
- User inbox: `final_report`만 수신합니다.


In [ ]:
outcome = run_delegation_a2a('재고를 조사해 발주 검토 보고서를 작성해줘.')
outcome['report']

### 2. A2A가 실제로 동작했는지 검증

단순히 예상 JSON을 출력하는 대신 message bus가 각 receiver handler를 실행하고 reply를 다음 receiver로 전달합니다. 아래 assertion은 경로, correlation ID, 읽기 전용 제약을 모두 검사합니다.

In [ ]:
verification = outcome['verification']
assert verification['passed']
assert verification['same_correlation_id']
assert verification['no_external_write']
assert verification['actual_hops'] == [
    ('orchestrator', 'researcher'),
    ('researcher', 'analyst_writer'),
    ('analyst_writer', 'user'),
]
verification

### 3. 전달 trace 검사

trace의 `payload`는 각각 task, evidence, final report입니다. 특히 researcher → analyst_writer hop이 없거나 message type이 달라지면 handler와 runtime 검증이 즉시 실패합니다.

In [ ]:
for event in outcome['trace']:
    message = event['message']
    print(f"{message['sender']} → {message['recipient']} | {message['message_type']} | {message['correlation_id']}")

### 4. 안전 경계 확인

이 notebook은 fixture 기반 읽기만 수행합니다. 실제 Harness의 DeepAgents, MCP Registry 또는 원격 A2A endpoint를 연결할 때는 transport만 교체하고, specialist별 도구 권한·approval·evidence packet 검증은 그대로 적용하세요.